# Stan in Colab

Fitting a graphical model with **Stan**, through the `mcmc` command line tool.

Every step below is one command: convert the graph to a model, fit it, check that it converged, draw the posterior, and package the run so it can be opened in the report app. Run the cells in order.

## Install

`mcmc` is a self-contained binary. `mcmc setup` then installs the Stan toolchain, which on a fresh Colab runtime takes several minutes.

In [ ]:
!curl -fsSL https://mcmcjs.github.io/install.sh | sh
import os

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!mcmc --version

In [ ]:
!mcmc setup --engine stan

## Your graph

1. In the editor, open the **Run** tab and press **Copy graph**.
2. Paste it between the quotes below, replacing the blank line.

Everything after this cell is generic: it reads whatever graph you paste. Run the notebook without pasting and it uses a small built-in example instead, so you can see the whole workflow first.

In [ ]:
PASTED = r'''

'''

EXAMPLE = r'''
{
  "name": "Pumps: Conjugate Gamma-Poisson Hierarchical Model",
  "elements": [
    {
      "id": "node_alpha",
      "name": "alpha",
      "type": "node",
      "nodeType": "stochastic",
      "position": {
        "x": 58,
        "y": 41
      },
      "distribution": "dexp",
      "param1": "1"
    },
    {
      "id": "node_beta",
      "name": "beta",
      "type": "node",
      "nodeType": "stochastic",
      "position": {
        "x": 218,
        "y": 41
      },
      "distribution": "dgamma",
      "param1": "0.1",
      "param2": "1.0"
    },
    {
      "id": "plate_i",
      "name": "Plate.i",
      "type": "node",
      "nodeType": "plate",
      "position": {
        "x": 190,
        "y": 330
      },
      "loopVariable": "i",
      "loopRange": "1:N"
    },
    {
      "id": "node_theta",
      "name": "theta",
      "type": "node",
      "nodeType": "stochastic",
      "position": {
        "x": 124,
        "y": 198
      },
      "parent": "plate_i",
      "distribution": "dgamma",
      "indices": "i",
      "param1": "alpha",
      "param2": "beta"
    },
    {
      "id": "node_t",
      "name": "t",
      "type": "node",
      "nodeType": "constant",
      "position": {
        "x": 256,
        "y": 198
      },
      "parent": "plate_i",
      "indices": "i"
    },
    {
      "id": "node_lambda",
      "name": "lambda",
      "type": "node",
      "nodeType": "deterministic",
      "position": {
        "x": 190,
        "y": 330
      },
      "parent": "plate_i",
      "equation": "theta[i] * t[i]",
      "indices": "i"
    },
    {
      "id": "node_x",
      "name": "x",
      "type": "node",
      "nodeType": "observed",
      "position": {
        "x": 190,
        "y": 462
      },
      "parent": "plate_i",
      "distribution": "dpois",
      "indices": "i",
      "observed": true,
      "param1": "lambda[i]"
    },
    {
      "id": "edge_alpha_theta",
      "type": "edge",
      "source": "node_alpha",
      "target": "node_theta",
      "relationshipType": "stochastic"
    },
    {
      "id": "edge_beta_theta",
      "type": "edge",
      "source": "node_beta",
      "target": "node_theta",
      "relationshipType": "stochastic"
    },
    {
      "id": "edge_theta_lambda",
      "type": "edge",
      "source": "node_theta",
      "target": "node_lambda",
      "relationshipType": "deterministic"
    },
    {
      "id": "edge_t_lambda",
      "type": "edge",
      "source": "node_t",
      "target": "node_lambda",
      "relationshipType": "deterministic"
    },
    {
      "id": "edge_lambda_x",
      "type": "edge",
      "source": "node_lambda",
      "target": "node_x",
      "relationshipType": "stochastic"
    }
  ],
  "dataContent": "{\n  \"data\": {\n    \"t\": [94.3, 15.7, 62.9, 126, 5.24, 31.4, 1.05, 1.05, 2.1, 10.5],\n    \"x\": [5, 1, 5, 14, 3, 19, 1, 1, 4, 22],\n    \"N\": 10\n  },\n  \"inits\": {\n    \"alpha\": 1,\n    \"beta\": 1\n  }\n}",
  "version": 1,
  "layout": {
    "showCodePanel": false,
    "codePanelWidth": 400,
    "codePanelHeight": 300,
    "showDataPanel": false,
    "dataPanelX": 40,
    "dataPanelY": 90,
    "dataPanelWidth": 300,
    "dataPanelHeight": 268
  }
}
'''

import json

graph = PASTED.strip() or EXAMPLE
print("using your pasted graph" if PASTED.strip() else "nothing pasted: using the built-in example")

with open("model.json", "w") as f:
    f.write(graph)

print("model:", json.loads(graph).get("name", "model"))

## The Stan model

What the graph becomes as code, plus the spec that runs it.

In [ ]:
!mcmc convert model.json --stan
!cat model.stan

## Fit

`mcmc run` does the whole workflow: it samples, checks convergence, and records the run so the later commands can find it.

In [ ]:
!mcmc run model.toml --chains 2 --draws 1000 --warmup 1000 --seed 42

## Did it converge?

R-hat near 1 and a healthy effective sample size per parameter. `mcmc diagnose` exits non-zero if it did not, so this is the cell to trust before reading the posterior.

In [ ]:
!mcmc summary
!mcmc diagnose

## Plots

Traces and ranks show the chains mixing; densities and the forest plot show the posterior itself.

In [ ]:
from IPython.display import SVG, display

for kind in ["trace","density","forest","rank"]:
    !mcmc plot --kind {kind} --format svg -o {kind}.svg
    print(kind)
    display(SVG(f"{kind}.svg"))

## Open the run in the report app

A run bundle holds the samples, the spec and the diagnostics in one file. Download it, then drop it into [the report app](https://mcmcjs.github.io/report/) to explore every parameter interactively.

In [ ]:
!mcmc export bundle -o run.mcmcrun.json

from google.colab import files  # skip this line outside Colab

files.download("run.mcmcrun.json")